In [ ]:
import pandas as pd
from pathlib import Path
from typing import Optional, Literal

In [66]:
def print_signed_pvalue_table_with_categories(
    csv_path: str | Path,
) -> None:
    """
    Print a compact LaTeX table with dataset-level and category-level coefficients.

    Rows are dataset/category combinations.
    Columns are terms/personas.

    Cells contain coefficients only:
        - sign = direction of coefficient
        - value = coefficient
        - bold = p < .05

    Expected CSV columns:
        dataset, category, term, coef, stderr, pvalue
    """

    overall_label = "Overall"
    p_threshold = 0.05

    df = pd.read_csv(csv_path)

    df["category"] = df["category"].fillna("").astype(str)
    df = df.sort_values(by=["dataset", "category"])

    default_terms = [
        "base",
        "static_short",
        "static_medium",
        "static_long",
        "dynamic_short",
        "dynamic_medium",
        "dynamic_long",
        "beginner_teacher",
        "intermediate_teacher",
        "expert_teacher",
    ]

    present_terms = set(df["term"])
    terms = [term for term in default_terms if term in present_terms]

    pretty_names = {
        "base": "base",
        "static_short": "short",
        "static_medium": "med.",
        "static_long": "long",
        "dynamic_short": "short",
        "dynamic_medium": "med.",
        "dynamic_long": "long",
        "beginner_teacher": "beg.",
        "intermediate_teacher": "int.",
        "expert_teacher": "exp.",
    }

    def latex_escape(text: str) -> str:
        return (
            str(text)
            .replace("\\", "\\textbackslash{}")
            .replace("_", "\\_")
            .replace("&", "\\&")
            .replace("%", "\\%")
            .replace("#", "\\#")
        )

    def format_coef(coef: float, p: float) -> str:
        if pd.isna(coef):
            return "{--}"

        formatted = f"{coef:.2f}"

        if pd.notna(p) and p < p_threshold:
            return rf"\bfseries {formatted}"

        return formatted

    # Mark dataset-level rows as Overall
    df["category_print"] = df["category"].where(
        df["category"].str.strip() != "",
        overall_label,
    )

    # Keep only selected terms/personas
    df = df[df["term"].isin(terms)]

    dataset_order = list(dict.fromkeys(df["dataset"]))

    rows = []

    for ds in dataset_order:
        ds_df = df[df["dataset"] == ds]

        # Overall first, if present
        if (ds_df["category_print"] == overall_label).any():
            rows.append((ds, overall_label))

        # Then subcategories
        subcats = list(
            dict.fromkeys(
                ds_df.loc[
                    ds_df["category_print"] != overall_label,
                    "category_print",
                ]
            )
        )

        for cat in subcats:
            rows.append((ds, cat))

    lookup = {
        (row["dataset"], row["category_print"], row["term"]): format_coef(
            row["coef"],
            row["pvalue"],
        )
        for _, row in df.iterrows()
    }

    # ll = Dataset, Category
    # one S column per coefficient
    col_spec = (
        "ll"
        + " "
        + " ".join(
            ["S[table-format=-1.2]" for _ in terms]
        )
    )

    print("\\begin{table}")
    print("\\centering")
    print("\\scriptsize")
    print("\\setlength{\\tabcolsep}{2.2pt}")
    print("\\renewcommand{\\arraystretch}{0.9}")
    print(f"\\begin{{tabular}}{{{col_spec}}}")
    print("\\toprule")

    # Grouped header
    # Column positions:
    # 1 Dataset, 2 Category, 3 base, 4-6 static, 7-9 dynamic, 10-12 teacher
    print(
        "& & "
        "& "
        "\\multicolumn{3}{c}{static} "
        "& \\multicolumn{3}{c}{dynamic} "
        "& \\multicolumn{3}{c}{teacher} \\\\"
    )
    print("\\cmidrule(lr){4-6} \\cmidrule(lr){7-9} \\cmidrule(lr){10-12}")

    headers = (
        ["Dataset", "Category"]
        + [f"{{{pretty_names.get(term, latex_escape(term))}}}" for term in terms]
    )

    print(" & ".join(headers) + " \\\\")
    print("\\midrule")

    previous_dataset = None

    for dataset_name, category_name in rows:
        if dataset_name != previous_dataset and previous_dataset is not None:
            print("\\midrule")

        dataset_cell = latex_escape(dataset_name) if dataset_name != previous_dataset else ""
        category_cell = latex_escape(category_name)

        values = [dataset_cell, category_cell]

        for term in terms:
            values.append(lookup.get((dataset_name, category_name, term), "{--}"))

        print(" & ".join(values) + " \\\\")

        previous_dataset = dataset_name

    print("\\bottomrule")
    print("\\end{tabular}")
    print("\\end{table}")

In [67]:
print_signed_pvalue_table_with_categories(
    "data/evaluation/tests/results_baseline.csv"
)

\begin{table}
\centering
\scriptsize
\setlength{\tabcolsep}{2.2pt}
\renewcommand{\arraystretch}{0.9}
\begin{tabular}{ll S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2]}
\toprule
& & & \multicolumn{3}{c}{static} & \multicolumn{3}{c}{dynamic} & \multicolumn{3}{c}{teacher} \\
\cmidrule(lr){4-6} \cmidrule(lr){7-9} \cmidrule(lr){10-12}
Dataset & Category & {base} & {short} & {med.} & {long} & {short} & {med.} & {long} & {beg.} & {int.} & {exp.} \\
\midrule
MATH & Overall & -0.10 & -0.03 & -0.04 & -0.05 & -0.07 & 0.02 & -0.03 & \bfseries -0.16 & -0.03 & -0.03 \\
 & Algebra & \bfseries -0.22 & -0.18 & -0.12 & \bfseries -0.23 & \bfseries -0.21 & -0.02 & -0.06 & \bfseries -0.28 & -0.12 & -0.17 \\
 & Counting \& Probability & -0.08 & -0.00 & -0.04 & -0.02 & -0.12 & -0.12 & -0.04 & 0.06 & -0.16 & -0.02 \\
 & Geometry & 0.02 & 0.03 & 0.09

In [68]:
def print_combined_effect_table(
    length_csv: str | Path,
    teacher_csv: str | Path,
    dynamic_csv: str | Path,
    datasets: Optional[list[str]] = None,
    caption: Optional[str] = None,
    label: Optional[str] = None,
    p_threshold: float = 0.05,
    overall_label: str = "Overall",
) -> None:
    """
    Print one compact LaTeX table combining:
      - length effect by mode from results_length.csv
      - teacher level effect from results_teacher.csv
      - static-vs-dynamic effect from results_static_vs_dynamic.csv

    Expected CSV structures:

    results_length.csv:
        dataset, category, term, mode, coef, stderr, pvalue

    results_teacher.csv:
        dataset, category, term, coef, stderr, pvalue

    results_static_vs_dynamic.csv:
        dataset, category, term, coef, stderr, pvalue

    Output columns:
        Dataset | Category | stat. | dyn. | com. | teach. level | dyn.

    Cells contain coefficients only.
    Bold coefficients indicate p < .05.
    """

    length_df = pd.read_csv(length_csv)
    teacher_df = pd.read_csv(teacher_csv)
    dynamic_df = pd.read_csv(dynamic_csv)

    def check_cols(df: pd.DataFrame, required: set[str], name: str) -> None:
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{name} is missing required columns: {missing}")

    check_cols(
        length_df,
        {"dataset", "category", "term", "mode", "coef", "pvalue"},
        "length_csv",
    )
    check_cols(
        teacher_df,
        {"dataset", "category", "term", "coef", "pvalue"},
        "teacher_csv",
    )
    check_cols(
        dynamic_df,
        {"dataset", "category", "term", "coef", "pvalue"},
        "dynamic_csv",
    )

    for df in [length_df, teacher_df, dynamic_df]:
        df["category"] = df["category"].fillna("").astype(str)

    if datasets is not None:
        length_df = length_df[length_df["dataset"].isin(datasets)]
        teacher_df = teacher_df[teacher_df["dataset"].isin(datasets)]
        dynamic_df = dynamic_df[dynamic_df["dataset"].isin(datasets)]

    # Keep only the relevant terms
    length_df = length_df[length_df["term"] == "length"].copy()
    teacher_df = teacher_df[teacher_df["term"] == "level"].copy()
    dynamic_df = dynamic_df[dynamic_df["term"] == "is_dynamic"].copy()

    def latex_escape(text: str) -> str:
        return (
            str(text)
            .replace("\\", "\\textbackslash{}")
            .replace("_", "\\_")
            .replace("&", "\\&")
            .replace("%", "\\%")
            .replace("#", "\\#")
        )

    def format_coef(coef: float, pvalue: float) -> str:
        """
        Format coefficient for siunitx S columns.
        Bold if p < p_threshold.
        """
        if pd.isna(coef):
            return "{--}"

        value = f"{coef:.2f}"

        if pd.notna(pvalue) and pvalue < p_threshold:
            return rf"\bfseries {value}"

        return value

    def add_category_print(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["category_print"] = df["category"].where(
            df["category"].str.strip() != "",
            overall_label,
        )
        return df

    length_df = add_category_print(length_df)
    teacher_df = add_category_print(teacher_df)
    dynamic_df = add_category_print(dynamic_df)

    records = []

    for _, row in length_df.iterrows():
        mode = str(row["mode"]).strip().lower()

        if mode == "static":
            effect = "length_static"
        elif mode == "dynamic":
            effect = "length_dynamic"
        elif mode == "combined":
            effect = "length_combined"
        else:
            effect = f"length_{mode}"

        records.append(
            {
                "dataset": row["dataset"],
                "category_print": row["category_print"],
                "effect": effect,
                "cell": format_coef(row["coef"], row["pvalue"]),
            }
        )

    for _, row in teacher_df.iterrows():
        records.append(
            {
                "dataset": row["dataset"],
                "category_print": row["category_print"],
                "effect": "teacher_level",
                "cell": format_coef(row["coef"], row["pvalue"]),
            }
        )

    for _, row in dynamic_df.iterrows():
        records.append(
            {
                "dataset": row["dataset"],
                "category_print": row["category_print"],
                "effect": "is_dynamic",
                "cell": format_coef(row["coef"], row["pvalue"]),
            }
        )

    combined = pd.DataFrame(records)

    effect_order = [
        "length_static",
        "length_dynamic",
        "length_combined",
        "teacher_level",
        "is_dynamic",
    ]

    effect_names = {
        "length_static": "stat.",
        "length_dynamic": "dyn.",
        "length_combined": "com.",
        "teacher_level": "teach. level",
        "is_dynamic": "dyn.",
    }

    # Preserve dataset order as it appears across files
    dataset_order = []

    for df in [length_df, teacher_df, dynamic_df]:
        for ds in df["dataset"]:
            if ds not in dataset_order:
                dataset_order.append(ds)

    rows = []

    for ds in dataset_order:
        ds_combined = combined[combined["dataset"] == ds]

        if (ds_combined["category_print"] == overall_label).any():
            rows.append((ds, overall_label))

        subcategories = list(
            dict.fromkeys(
                ds_combined.loc[
                    ds_combined["category_print"] != overall_label,
                    "category_print",
                ]
            )
        )

        for cat in subcategories:
            rows.append((ds, cat))

    lookup = {
        (row["dataset"], row["category_print"], row["effect"]): row["cell"]
        for _, row in combined.iterrows()
    }

    # Two text columns + one decimal-aligned S column per effect
    col_spec = (
        "ll "
        + " ".join(["S[table-format=-1.2]" for _ in effect_order])
    )

    print("\\begin{table}[htbp]")
    print("\\centering")
    print("\\scriptsize")
    print("\\setlength{\\tabcolsep}{3pt}")
    print("\\renewcommand{\\arraystretch}{0.9}")
    print(f"\\begin{{tabular}}{{{col_spec}}}")
    print("\\toprule")

    headers = (
        ["Dataset", "Category"]
        + [f"{{{effect_names[e]}}}" for e in effect_order]
    )
    print(" & ".join(headers) + " \\\\")

    print("\\midrule")

    previous_dataset = None

    for dataset_name, category_name in rows:
        if dataset_name != previous_dataset and previous_dataset is not None:
            print("\\midrule")

        dataset_cell = latex_escape(dataset_name) if dataset_name != previous_dataset else ""
        category_cell = latex_escape(category_name)

        values = [dataset_cell, category_cell]

        for effect in effect_order:
            values.append(lookup.get((dataset_name, category_name, effect), "{--}"))

        print(" & ".join(values) + " \\\\")

        previous_dataset = dataset_name

    print("\\bottomrule")
    print("\\end{tabular}")

    if caption is None:
        caption = (
            "Coefficients for length, teacher-level, and static-versus-dynamic effects "
            "by dataset and category. Bold values indicate $p < .05$."
        )

    print(f"\\caption{{{caption}}}")

    if label is not None:
        print(f"\\label{{{label}}}")

    print("\\end{table}")

In [69]:
print_combined_effect_table(
    length_csv="data/evaluation/tests/results_length.csv",
    teacher_csv="data/evaluation/tests/results_teacher.csv",
    dynamic_csv="data/evaluation/tests/results_static_vs_dynamic.csv",
    label="tab:length_teacher_dynamic_effects",
)

\begin{table}[htbp]
\centering
\scriptsize
\setlength{\tabcolsep}{3pt}
\renewcommand{\arraystretch}{0.9}
\begin{tabular}{ll S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2] S[table-format=-1.2]}
\toprule
Dataset & Category & {stat.} & {dyn.} & {com.} & {teach. level} & {dyn.} \\
\midrule
mmlu-pro & Overall & \bfseries 0.03 & \bfseries 0.02 & \bfseries 0.02 & 0.02 & 0.02 \\
 & biology & 0.02 & 0.02 & 0.02 & 0.04 & 0.01 \\
 & business & \bfseries 0.07 & 0.01 & 0.04 & 0.04 & -0.06 \\
 & chemistry & 0.03 & 0.02 & 0.02 & 0.01 & -0.02 \\
 & computer science & 0.03 & 0.00 & 0.02 & -0.01 & -0.06 \\
 & economics & 0.04 & 0.01 & 0.03 & 0.04 & -0.06 \\
 & engineering & -0.00 & 0.01 & 0.00 & -0.04 & 0.03 \\
 & health & 0.03 & 0.04 & 0.02 & \bfseries 0.10 & 0.10 \\
 & history & 0.02 & 0.03 & 0.02 & 0.03 & 0.08 \\
 & law & 0.02 & 0.03 & 0.02 & 0.04 & -0.02 \\
 & math & -0.01 & -0.00 & -0.01 & 0.01 & 0.06 \\
 & other & 0.05 & 0.03 & 0.04 & -0.02 & 0.03 \\
 & philoso

In [59]:
def print_combined_lr_table_aligned(
    length_csv: str | Path,
    teacher_csv: str | Path,
    dynamic_csv: str | Path,
    datasets: Optional[list[str]] = None,
    caption: Optional[str] = None,
    label: Optional[str] = None,
    p_threshold: float = 0.05,
    lr_digits: int = 2,
    p_digits: int = 3,
    use_less_than: bool = True,
    overall_label: str = "Overall",
) -> None:
    """
    Print one compact LaTeX table combining likelihood-ratio test results.

    This version uses siunitx S columns so that LR statistics and p-values
    are aligned at the decimal point.

    Expected CSV structures:

    results_length_model_lr.csv:
        dataset, category, lr_stat, diff, pvalue, mode

    results_teacher_model_lr.csv:
        dataset, category, lr_stat, diff, pvalue

    results_static_vs_dynamic_model_lr.csv:
        dataset, category, lr_stat, diff, pvalue

    Output:
        Dataset | Category |
        Length stat. LR | Length stat. p |
        Length dyn. LR  | Length dyn. p |
        Length comb. LR | Length comb. p |
        Teacher level LR | Teacher level p |
        Stat. vs dyn. LR | Stat. vs dyn. p

    Bold values indicate p < p_threshold.
    """

    length_df = pd.read_csv(length_csv)
    teacher_df = pd.read_csv(teacher_csv)
    dynamic_df = pd.read_csv(dynamic_csv)

    def check_cols(df: pd.DataFrame, required: set[str], name: str) -> None:
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{name} is missing required columns: {missing}")

    check_cols(
        length_df,
        {"dataset", "category", "lr_stat", "pvalue", "mode"},
        "length_csv",
    )
    check_cols(
        teacher_df,
        {"dataset", "category", "lr_stat", "pvalue"},
        "teacher_csv",
    )
    check_cols(
        dynamic_df,
        {"dataset", "category", "lr_stat", "pvalue"},
        "dynamic_csv",
    )

    for df in [length_df, teacher_df, dynamic_df]:
        df["category"] = df["category"].fillna("").astype(str)

    if datasets is not None:
        length_df = length_df[length_df["dataset"].isin(datasets)]
        teacher_df = teacher_df[teacher_df["dataset"].isin(datasets)]
        dynamic_df = dynamic_df[dynamic_df["dataset"].isin(datasets)]

    def latex_escape(text: str) -> str:
        return (
            str(text)
            .replace("\\", "\\textbackslash{}")
            .replace("_", "\\_")
            .replace("&", "\\&")
            .replace("%", "\\%")
            .replace("#", "\\#")
        )

    def format_pvalue(pvalue: float) -> str:
        """
        Format p-values for siunitx S columns.

        Examples:
            0.068 -> .068
            0.0000001 -> <.001
        """
        if pd.isna(pvalue):
            return "--"

        value = f"{pvalue:.{p_digits}f}"

        # thesis style: .068 instead of 0.068
        if value.startswith("0."):
            value = value[1:4]
        else:
            value = value[:3]
        if pvalue < p_threshold:
            return rf"\bfseries {value}"

        return value

    def format_lr(lr_stat: float, pvalue: float) -> str:
        """
        Format LR statistic. Bold if the corresponding p-value is significant.
        """
        if pd.isna(lr_stat) or pd.isna(pvalue):
            return "--"

        value = f"{lr_stat:.1f}"
        return value

    def format_p(pvalue: float) -> str:
        """
        Format p-value. Bold if significant.
        """
        if pd.isna(pvalue):
            return "--"

        value = format_pvalue(pvalue)

        if pvalue < p_threshold:
            return rf"\bfseries {value}"

        return value

    records = []

    for _, row in length_df.iterrows():
        mode = str(row["mode"]).strip().lower()

        if mode == "static":
            effect = "length_static"
        elif mode == "dynamic":
            effect = "length_dynamic"
        elif mode == "combined":
            effect = "length_combined"
        else:
            effect = f"length_{mode}"

        records.append(
            {
                "dataset": row["dataset"],
                "effect": effect,
                "lr": format_lr(row["lr_stat"], row["pvalue"]),
                "p": format_p(row["pvalue"]),
            }
        )

    for _, row in teacher_df.iterrows():
        records.append(
            {
                "dataset": row["dataset"],
                "effect": "teacher_level",
                "lr": format_lr(row["lr_stat"], row["pvalue"]),
                "p": format_p(row["pvalue"]),
            }
        )

    for _, row in dynamic_df.iterrows():
        records.append(
            {
                "dataset": row["dataset"],
                "effect": "static_vs_dynamic",
                "lr": format_lr(row["lr_stat"], row["pvalue"]),
                "p": format_p(row["pvalue"]),
            }
        )

    combined = pd.DataFrame(records)

    effect_order = [
        "length_static",
        "length_dynamic",
        "length_combined",
        "teacher_level",
        "static_vs_dynamic",
    ]

    effect_names = {
        "length_static": "Length stat.",
        "length_dynamic": "Length dyn.",
        "length_combined": "Length comb.",
        "teacher_level": "Teacher level",
        "static_vs_dynamic": "Stat. vs dyn.",
    }

    # Preserve dataset order as it appears across the input files
    dataset_order = []
    for df in [length_df, teacher_df, dynamic_df]:
        for ds in df["dataset"]:
            if ds not in dataset_order:
                dataset_order.append(ds)

    rows = []

    for ds in dataset_order:
        rows.append(ds)

    lookup = {}

    for _, row in combined.iterrows():
        key = (row["dataset"], row["effect"])
        lookup[(key, "lr")] = row["lr"]
        lookup[(key, "p")] = row["p"]

    # Two S columns per effect: one for LR, one for p
    col_spec = (
        "ll"
        + " "
        + " ".join(
            [
                f"S[table-format=3.{lr_digits}] S[table-format=<1.{p_digits}]"
                for _ in effect_order
            ]
        )
    )

    print("\\begin{table}[htbp]")
    print("\\centering")
    print("\\scriptsize")
    print("\\setlength{\\tabcolsep}{2.2pt}")
    print("\\renewcommand{\\arraystretch}{0.9}")
    print(f"\\begin{{tabular}}{{{col_spec}}}")
    print("\\toprule")

    # First header row
    header = "Dataset"
    for effect in effect_order:
        header += f" & \\multicolumn{{2}}{{c}}{{{effect_names[effect]}}}"
    header += r" \\"
    print(header)

    # cmidrules below grouped headers
    cmidrules = []
    start_col = 3
    for _ in effect_order:
        end_col = start_col + 1
        cmidrules.append(f"\\cmidrule(lr){{{start_col}-{end_col}}}")
        start_col += 2

    print(" ".join(cmidrules))

    # Second header row
    subheader = "& & " + " & ".join(["{$LR$} & {$p$}" for _ in effect_order])
    subheader += r" \\"
    print(subheader)

    print("\\midrule")

    previous_dataset = None

    for dataset_name in rows:
        dataset_cell = latex_escape(dataset_name) if dataset_name != previous_dataset else ""
        values = [dataset_cell]

        for effect in effect_order:
            key = (dataset_name, effect)
            values.append(lookup.get((key, "lr"), "--"))
            values.append(lookup.get((key, "p"), "--"))

        print(" & ".join(values) + r" \\")

        previous_dataset = dataset_name

    print("\\bottomrule")
    print("\\end{tabular}")

    if caption is None:
        caption = (
            "Likelihood-ratio tests for model-dependent effects. "
            "Cells report likelihood-ratio statistics and p-values. "
            "Bold values indicate $p < .05$."
        )

    print(f"\\caption{{{caption}}}")

    if label is not None:
        print(f"\\label{{{label}}}")

    print("\\end{table}")

In [60]:
print_combined_lr_table_aligned(
    length_csv="data/evaluation/tests/results_length_model_lr.csv",
    teacher_csv="data/evaluation/tests/results_teacher_model_lr.csv",
    dynamic_csv="data/evaluation/tests/results_static_vs_dynamic_model_lr.csv",
    label="tab:model_dependency_lr_tests",
)

\begin{table}[htbp]
\centering
\scriptsize
\setlength{\tabcolsep}{2.2pt}
\renewcommand{\arraystretch}{0.9}
\begin{tabular}{ll S[table-format=3.2] S[table-format=<1.3] S[table-format=3.2] S[table-format=<1.3] S[table-format=3.2] S[table-format=<1.3] S[table-format=3.2] S[table-format=<1.3] S[table-format=3.2] S[table-format=<1.3]}
\toprule
Dataset & \multicolumn{2}{c}{Length stat.} & \multicolumn{2}{c}{Length dyn.} & \multicolumn{2}{c}{Length comb.} & \multicolumn{2}{c}{Teacher level} & \multicolumn{2}{c}{Stat. vs dyn.} \\
\cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-8} \cmidrule(lr){9-10} \cmidrule(lr){11-12}
& & {$LR$} & {$p$} & {$LR$} & {$p$} & {$LR$} & {$p$} & {$LR$} & {$p$} & {$LR$} & {$p$} \\
\midrule
mmlu-pro & 55.9 & \bfseries \bfseries .00 & 56.7 & \bfseries \bfseries .00 & 65.4 & \bfseries \bfseries .00 & 38.0 & \bfseries \bfseries .00 & 87.6 & \bfseries \bfseries .00 \\
alpaca & 137.8 & \bfseries \bfseries .00 & 139.6 & \bfseries \bfseries .00 & 62.0 & \bfseries \bfs

In [86]:
def print_accuracy_table_for_dataset(
    csv_path: str | Path,
    dataset_name: Optional[str] = None,
    metric: Literal["accuracy", "win_rate"] = "accuracy",
    score_col: str = "score",
    model_col: str = "model",
    persona_col: str = "persona",
    digits: int = 3,
    caption: Optional[str] = None,
    label: Optional[str] = None,
) -> None:
    """
    Print one LaTeX table for one dataset.

    Rows are models.
    Columns are personas.
    Cells are average accuracy scores or win rates.
    The final row reports the average across models.

    Expected CSV columns:
        model, persona, score

    Args:
        metric:
            "accuracy":
                Uses the mean of the score column.
                Appropriate for binary 0/1 accuracy datasets.

            "win_rate":
                Converts score to 1 if score == 2, otherwise 0.
                Appropriate for Alpaca/FLORES-style pairwise judgments
                where score == 2 means the persona wins.
    """

    df = pd.read_csv(csv_path)

    required_cols = {model_col, persona_col, score_col}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")

    if dataset_name is None:
        dataset_name = Path(csv_path).stem

    df = df.copy()

    if metric == "accuracy":
        value_col = score_col
    elif metric == "win_rate":
        value_col = "_win"
        df[value_col] = (df[score_col] == 2).astype(int)
    else:
        raise ValueError("metric must be either 'accuracy' or 'win_rate'.")

    persona_order = [
        "no",
        "base",
        "helpful",
        "static_short",
        "static_medium",
        "static_long",
        "dynamic_short",
        "dynamic_medium",
        "dynamic_long",
        "beginner_teacher",
        "intermediate_teacher",
        "expert_teacher",
    ]

    model_order = [
        "gemma-3-1b-it",
        "gemma-3-4b-it",
        "llama-3-2-1b-instruct",
        "llama-3-2-3b-instruct",
        "qwen3-0-6b",
        "qwen3-4b",
        "gpt-4-1-nano",
        "gpt-5-nano",
    ]

    present_personas = set(df[persona_col])
    personas = [p for p in persona_order if p in present_personas]
    personas += sorted(present_personas - set(personas))

    present_models = set(df[model_col])
    models = [m for m in model_order if m in present_models]
    models += sorted(present_models - set(models))

    pretty_personas = {
        "no": "No",
        "helpful": "Help.",
        "base": "Base",
        "static_short": "Short",
        "static_medium": "Med.",
        "static_long": "Long",
        "dynamic_short": "Short",
        "dynamic_medium": "Med.",
        "dynamic_long": "Long",
        "beginner_teacher": "Beg.",
        "intermediate_teacher": "Int.",
        "expert_teacher": "Exp.",
    }

    pretty_models = {
        "gemma-3-1b-it": "Gemma 3 1B",
        "gemma-3-4b-it": "Gemma 3 4B",
        "llama-3-2-1b-instruct": "Llama 3.2 1B",
        "llama-3-2-3b-instruct": "Llama 3.2 3B",
        "qwen3-0-6b": "Qwen3 0.6B",
        "qwen3-4b": "Qwen3 4B",
        "gpt-4-1-nano": "GPT-4.1 nano",
        "gpt-5-nano": "GPT-5 nano",
    }

    def latex_escape(text: str) -> str:
        return (
            str(text)
            .replace("\\", "\\textbackslash{}")
            .replace("_", "\\_")
            .replace("&", "\\&")
            .replace("%", "\\%")
            .replace("#", "\\#")
        )

    def format_score(x: float) -> str:
        if pd.isna(x):
            return "{--}"

        value = f"{x:.2f}"

        # thesis style: .734 instead of 0.734
        if value.startswith("0."):
            value = value[1:]

        return value

    pivot = (
        df.groupby([model_col, persona_col])[value_col]
        .mean()
        .unstack(persona_col)
        .reindex(index=models, columns=personas)
    )

    average_row = pivot.mean(axis=0)
    pivot.loc["Average"] = average_row

    col_spec = (
        "l "
        + " ".join([f"S[table-format=1.2]" for _ in personas])
    )

    print("\\begin{table}[htbp]")
    print("\\centering")
    print("\\scriptsize")
    print("\\setlength{\\tabcolsep}{5pt}")
    print(f"\\begin{{tabular}}{{{col_spec}}}")
    print("\\toprule")
    print(
        "& & & &"
        "\\multicolumn{3}{c}{Static} "
        "& \\multicolumn{3}{c}{Dynamic} "
        "& \\multicolumn{3}{c}{Teacher} \\\\"
    )
    print("\\cmidrule(lr){5-7} \\cmidrule(lr){8-10} \\cmidrule(lr){11-13}")

    headers = ["Model"] + [
        f"{{{pretty_personas.get(p, latex_escape(p))}}}" for p in personas
    ]
    print(" & ".join(headers) + " \\\\")

    print("\\midrule")

    for model in pivot.index:
        if model == "Average":
            print("\\midrule")
            model_cell = "\\textbf{Average}"
        else:
            model_cell = latex_escape(pretty_models.get(model, model))

        values = [model_cell]

        for persona in personas:
            values.append(format_score(pivot.loc[model, persona]))

        print(" & ".join(values) + " \\\\")

    print("\\bottomrule")
    print("\\end{tabular}")

    if caption is None:
        if metric == "win_rate":
            caption = (
                f"Average win rates by model and persona for {latex_escape(dataset_name)}. "
                f"A win is defined as a score of 2."
            )
        else:
            caption = (
                f"Average accuracy by model and persona for {latex_escape(dataset_name)}."
            )

    print(f"\\caption{{{caption}}}")

    if label is not None:
        print(f"\\label{{{label}}}")

    print("\\end{table}")

In [80]:
print_accuracy_table_for_dataset(
    "data/evaluation/MATH.csv",
    dataset_name="MATH",
    metric="accuracy",
    label="tab:math_model_persona_accuracy",
)

\begin{table}[htbp]
\centering
\scriptsize
\setlength{\tabcolsep}{2.2pt}
\renewcommand{\arraystretch}{0.9}
\begin{tabular}{l S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2]}
\toprule
& & & &\multicolumn{3}{c}{Static} & \multicolumn{3}{c}{Dynamic} & \multicolumn{3}{c}{Teacher} \\
\cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
Model & {No} & {Base} & {Help.} & {Short} & {Med.} & {Long} & {Short} & {Med.} & {Long} & {Beg.} & {Int.} & {Exp.} \\
\midrule
Gemma 3 1B & .14 & .19 & .18 & .19 & .20 & .15 & .20 & .22 & .18 & .17 & .22 & .19 \\
Gemma 3 4B & .52 & .53 & .56 & .58 & .56 & .60 & .57 & .59 & .61 & .51 & .64 & .63 \\
Llama 3.2 1B & .25 & .23 & .25 & .20 & .21 & .22 & .21 & .22 & .21 & .19 & .18 & .21 \\
Llama 3.2 3B & .32 & .32 & .33 & .31 & .31 & .35 & .34 & .35 & .36 & .35 & .36 & 

In [81]:
print_accuracy_table_for_dataset(
    "data/evaluation/mmlu-pro.csv",
    dataset_name="MMLU-Pro",
    metric="accuracy",
    label="tab:mmlu_pro_model_persona_accuracy",
)

\begin{table}[htbp]
\centering
\scriptsize
\setlength{\tabcolsep}{2.2pt}
\renewcommand{\arraystretch}{0.9}
\begin{tabular}{l S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2]}
\toprule
& & & &\multicolumn{3}{c}{Static} & \multicolumn{3}{c}{Dynamic} & \multicolumn{3}{c}{Teacher} \\
\cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
Model & {No} & {Base} & {Help.} & {Short} & {Med.} & {Long} & {Short} & {Med.} & {Long} & {Beg.} & {Int.} & {Exp.} \\
\midrule
Gemma 3 1B & .13 & .12 & .13 & .12 & .11 & .11 & .11 & .12 & .12 & .08 & .13 & .11 \\
Gemma 3 4B & .01 & .01 & .01 & .01 & .01 & .01 & .01 & .01 & .01 & .03 & .02 & .01 \\
Llama 3.2 1B & .03 & .06 & .06 & .05 & .04 & .04 & .05 & .04 & .05 & .03 & .03 & .03 \\
Llama 3.2 3B & .29 & .28 & .28 & .28 & .29 & .29 & .32 & .32 & .33 & .30 & .32 & 

In [85]:
print_accuracy_table_for_dataset(
    "data/evaluation/ifbench.csv",
    dataset_name="IFBench",
    metric="accuracy",
    label="tab:ifbench_model_persona_accuracy",
)

\begin{table}[htbp]
\centering
\scriptsize
\setlength{	abcolsep}{5pt}
\begin{tabular}{l S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2]}
\toprule
& & & &\multicolumn{3}{c}{Static} & \multicolumn{3}{c}{Dynamic} & \multicolumn{3}{c}{Teacher} \\
\cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
Model & {No} & {Base} & {Help.} & {Short} & {Med.} & {Long} & {Short} & {Med.} & {Long} & {Beg.} & {Int.} & {Exp.} \\
\midrule
Gemma 3 1B & .19 & .21 & .21 & .19 & .19 & .16 & .20 & .17 & .18 & .18 & .17 & .16 \\
Gemma 3 4B & .25 & .26 & .26 & .26 & .22 & .20 & .23 & .22 & .20 & .18 & .18 & .21 \\
Llama 3.2 1B & .19 & .21 & .20 & .21 & .21 & .21 & .22 & .20 & .21 & .24 & .22 & .22 \\
Llama 3.2 3B & .25 & .28 & .26 & .27 & .22 & .24 & .28 & .28 & .29 & .26 & .27 & .28 \\
Qwen3 0.6B & .15 & .14 & .13 &

In [88]:
print_accuracy_table_for_dataset(
    "data/evaluation/alpaca.csv",
    dataset_name="Alpaca",
    metric="win_rate",
    label="tab:alpaca_model_persona_win_rate",
)

\begin{table}[htbp]
\centering
\scriptsize
\setlength{\tabcolsep}{5pt}
\begin{tabular}{l S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2]}
\toprule
& & & &\multicolumn{3}{c}{Static} & \multicolumn{3}{c}{Dynamic} & \multicolumn{3}{c}{Teacher} \\
\cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
Model & {No} & {Base} & {Short} & {Med.} & {Long} & {Short} & {Med.} & {Long} & {Beg.} & {Int.} & {Exp.} \\
\midrule
Gemma 3 1B & .50 & .43 & .26 & .21 & .28 & .63 & .59 & .60 & .14 & .36 & .32 \\
Gemma 3 4B & .54 & .41 & .14 & .13 & .19 & .74 & .67 & .64 & .14 & .40 & .37 \\
Llama 3.2 1B & .48 & .50 & .45 & .40 & .51 & .63 & .63 & .66 & .42 & .54 & .51 \\
Llama 3.2 3B & .50 & .47 & .37 & .39 & .57 & .75 & .73 & .72 & .31 & .52 & .55 \\
Qwen3 0.6B & .51 & .48 & .42 & .42 & .44 & .67 & .66 & .69 & .46 & .67 & .60 \\
Qwe

In [89]:
print_accuracy_table_for_dataset(
    "data/evaluation/flores.csv",
    dataset_name="FLORES+",
    metric="win_rate",
    label="tab:flores_model_persona_win_rate",
)

\begin{table}[htbp]
\centering
\scriptsize
\setlength{\tabcolsep}{5pt}
\begin{tabular}{l S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2] S[table-format=1.2]}
\toprule
& & & &\multicolumn{3}{c}{Static} & \multicolumn{3}{c}{Dynamic} & \multicolumn{3}{c}{Teacher} \\
\cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}
Model & {No} & {Base} & {Short} & {Med.} & {Long} & {Short} & {Med.} & {Long} & {Beg.} & {Int.} & {Exp.} \\
\midrule
Gemma 3 1B & .47 & .51 & .56 & .57 & .58 & .64 & .61 & .62 & .21 & .43 & .62 \\
Gemma 3 4B & .47 & .52 & .56 & .49 & .44 & .65 & .66 & .62 & .11 & .26 & .49 \\
Llama 3.2 1B & .41 & .56 & .59 & .58 & .58 & .65 & .63 & .63 & .52 & .58 & .60 \\
Llama 3.2 3B & .37 & .49 & .56 & .57 & .59 & .64 & .61 & .61 & .25 & .40 & .55 \\
Qwen3 0.6B & .44 & .46 & .50 & .47 & .49 & .54 & .53 & .52 & .45 & .48 & .49 \\
Qwe